In [15]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

from pathlib import Path
import os
import sys
from datetime import datetime, date
import time

import math

In [16]:
def wagner_whitin(d, f, h):
    """
    Implementação do algoritmo de programação dinâmica de Wagner-Whitin.
    
    Parâmetros
    ----------
    d : lista de demandas por período (1..T)
    f : lista de custos de preparação (setup) por período
    h : lista de custos de estocagem por período
    
    Retorna
    -------
    C : custo mínimo total
    prod : lista dos períodos em que é feita a produção
    """
    
    T = len(d)

    # Custo mínimo até o período t
    F = [0] + [math.inf] * T  
    
    # Para reconstrução da solução
    pred = [-1] * (T + 1)

    # Pré-calcula custos de produzir no período j para atender períodos j..t
    cost = [[0]*(T+1) for _ in range(T+1)]

    for j in range(1, T+1):
        for t in range(j, T+1):
            estoque = 0
            total = f[j-1]  # custo de preparação no período j
            for u in range(j, t+1):
                total += h[u-1] * estoque
                estoque += d[u-1]
            cost[j][t] = total

    # Programação dinâmica
    for t in range(1, T+1):
        for j in range(1, t+1):
            if F[j-1] + cost[j][t] < F[t]:
                F[t] = F[j-1] + cost[j][t]
                pred[t] = j-1

    # Reconstrói a solução (períodos de produção)
    prod_periods = []
    t = T
    while t > 0:
        j = pred[t]
        prod_periods.append(j + 1)
        t = j

    prod_periods.reverse()

    return F[T], prod_periods

In [17]:
# ---------------------------
# EXEMPLO DO LIVRO (Wolsey)
# ---------------------------

n = 4
d = [2, 4, 5, 1] 
p = [3, 3, 3, 3] 
h = [1, 2, 1, 1]
f = [12, 20, 16, 8]

optimal, production = wagner_whitin(d, p, h)

print("Optimal:", optimal)
print("Production:", production)


Optimal: 12
Production: [1, 2, 3, 4]


In [18]:
def uls_dp(demanda, cproducao, csetup):
    """
    Resolve o problema de dimensionamento de lotes usando programação dinâmica.

    Args:
        cdemanda (list): Lista com a demanda de cada período.
        cproducao (list): Lista com o custo de produção por unidade de cada período.
        csetup (list): Lista com o custo fixo de setup de cada período.

    Returns:
        tuple: Um tupla contendo o custo total mínimo e a política de produção.
    """
    n = len(demanda)
    # dp[i] armazena o custo mínimo para os primeiros i períodos
    dp = [float('inf')] * (n + 1)
    dp[0] = 0
    # politica[i] armazena a decisão ótima (produzir no período j para atender o período i)
    politica = [0] * (n + 1)

    for i in range(1, n + 1):
        cacumulado = 0
        for j in range(i, 0, -1):
            # Calcula o custo para produzir no período j e atender a demanda até o período i
            cacumulado += demanda[j - 1] * cproducao[j - 1]
            ctotal = dp[j - 1] + csetup[j - 1] + cacumulado

            if ctotal < dp[i]:
                dp[i] = ctotal
                politica[i] = j

    # Reconstruindo a política de produção
    producao = []
    i = n
    while i > 0:
        j = politica[i]
        producao.append((j, i))
        i = j - 1

    producao.reverse()
    return dp[n], producao

In [19]:
# Exemplo de uso
demanda = [10, 15, 20, 12, 18]
cproducao = [2, 3, 4, 2, 3]
csetup = [50, 60, 55, 65, 58]

optimal, production = uls_dp(demanda, cproducao, csetup)

print(f"Optimal: {optimal}")
print(f"Production: {production}")

Optimal: 273
Production: [(1, 5)]


In [20]:
p = [3, 3, 3, 3] 
h = [1, 2, 1, 1]

# Exemplo de uso
demanda = [2, 4, 5, 1]
csetup = [12, 20, 16, 8]

cproducao = []
for t in range(len(demanda)):
    a = p[t]
    for i in range(t,n):
        a += h[i]
    cproducao.append(a)
print(cproducao)

[8, 7, 5, 4]


In [21]:
optimal, production = uls_dp(demanda, cproducao, csetup)

print(f"Optimal: {optimal}")
print(f"Production: {production}")

Optimal: 85
Production: [(1, 4)]


In [22]:
folder_path = "../../data/uls/"

In [24]:
def readdata(datafile):

    path = os.path.join(folder_path, datafile)
    with open(path, 'r') as file:
        linhas = file.readlines()

    # remove linha vazia inicial e elimina os "\n" de cada linha
    linhas = [a.strip() for a in linhas if a.strip() != ""]

    # lendo o tamanho da instancia
    N = int(float(linhas[0]))  # primeira linha é o número de períodos

    # definindo vetores
    H = np.zeros(N)
    P = np.zeros(N)
    F = np.zeros(N)
    D = np.zeros(N)

    # lendo e armazenando dados
    F[0] = float(linhas[1])
    for i in range(1, N):
        F[i] = F[0]

    H[0] = float(linhas[2])
    for i in range(1, N):
        H[i] = H[0]

    P[0] = float(linhas[3])
    for i in range(1, N):
        P[i] = P[0]

    # lendo demandas
    demandas = linhas[4].split()
    for i in range(min(N, len(demandas))):
        D[i] = float(demandas[i])

    return N, H, P, F, D

In [25]:
def uls_dynamicp(N, H, P, F, D):
    """
    Programação dinâmica
    Baseada na formulação de Wagner-Whitin
    """
    
    # pré-calcular demandas acumuladas
    S = np.zeros(N + 1)
    for i in range(1, N + 1):
        S[i] = S[i - 1] + D[i - 1]

    # F[k] = custo mínimo para os primeiros k períodos
    F_dp = np.full(N + 1, np.inf)
    pred = np.full(N + 1, -1, dtype=int)
    F_dp[0] = 0.0

    for k in range(1, N + 1):
        for j in range(0, k):  # j = 0, 1, ..., k-1
            # qroduzir no período j+1 para atender j+1 até k
            t = j + 1

            # quantidade a produzir
            quantity = S[k] - S[j]

            # custo de produção
            production_cost = F[t-1] + P[t-1] * quantity

            # custo de estoque
            # para cada período m de t+1 até k, a demanda D[m-1] fica em estoque
            # desde o período t até m-1
            holding_cost = 0.0

            # simular o acúmulo de estoque período a período
            inventory = 0.0
            for period in range(t, k + 1):
                if period == t:
                    # no período de produção, adiciona a quantidade produzida
                    inventory += quantity

                # subtrai a demanda do período
                inventory -= D[period - 1]

                # adiciona custo de estoque se inventory > 0
                if period < k:  # estoque no final do período
                    holding_cost += H[period - 1] * inventory


            total_cost = F_dp[j] + production_cost + holding_cost

            if total_cost < F_dp[k]:
                F_dp[k] = total_cost
                pred[k] = t

    return F_dp[N], F_dp, pred, S

In [26]:
def recover_plan(pred, S, D, N):

    """updated solution"""
    x = np.zeros(N)
    y = np.zeros(N)

    k = N
    production_periods = []

    while k > 0:
        t = pred[k]
        production_periods.append(t)
        x[t-1] = S[k] - S[t-1]
        y[t-1] = 1
        k = t - 1

    # calcular estoques
    s = np.zeros(N)
    current_inv = 0.0
    for i in range(N):
        current_inv += x[i] - D[i]
        s[i] = current_inv

    return x, y, s

In [27]:
def main_dynamic(N, H, P, F, D):

    try:
        objval, F_dp, pred, S = uls_dynamicp(N, H, P, F, D)

        # recuperar plano (opcional para o resumo, mas útil para debug)
        x, y, s = recover_plan(pred, S, D, N)

        # mostrar plano (removido para não poluir a saída com muitas instâncias)
        # print("\nPlano de produção:")
        # for i in range(N):
        #     if x[i] > 0.001:
        #         print(f"  Período {i+1}: produzir {x[i]:.1f}, estoque final: {s[i]:.1f}")

        #return datafile, N, F[0], P[0], H[0], sum(D), objval, #exec_time
        return objval


    except Exception as e:
        print(f"Erro ao processar {e}")

In [28]:
def uls_std_mip(N, PP, FP, HP, D):
	try:

		SD = (np.zeros((N,N))).tolist()
		for  i in range(N):
			SD[i][i] = D[i]
			for j in range(i+1, N):
				SD[i][j] = SD[i][j-1] + D[j]

		# create model
		model = gp.Model("lsr_std_mip")

		# create variables
		xp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="xp")
		yp = model.addVars(list(range(N)), vtype=GRB.BINARY, name="yp")
		sp = model.addVars(list(range(N)), vtype=GRB.CONTINUOUS, name="sp")
		
		model.update()

		# set objective
		model.setObjective(gp.quicksum(
			PP[i]*xp[i] + HP[i]*sp[i] + FP[i]*yp[i] for i in range(N)), sense = GRB.MINIMIZE)

		# add constraints
		model.addConstr(xp[0] - sp[0] == D[0])
		model.addConstrs(sp[i-1] + xp[i] - sp[i] == D[i] for i in range(N) if i > 0 )
		
		model.addConstrs(xp[i] - yp[i]*SD[i][N-1] <= 0 for i in range(N))		
		
		# export .lp
		#model.write("instance.lp")

		# parameters 
		model.setParam(GRB.Param.TimeLimit, 360)
		model.setParam(GRB.Param.MIPGap, 0.0001)
		model.setParam(GRB.Param.Threads, 1)
		#model.setParam(GRB.Param.Cuts, 0)
		#model.setParam(GRB.Param.Presolve, 0)
		#model.setParam(GRB.Param.SolutionLimit, 1)
		# To see detailed logs (default)
		model.setParam('OutputFlag', 0) # 1: Enabled, 0: Disabled [2, 9]
		model.setParam('LogToConsole', 0) # 1: To console, 0: Suppress [2]
		#model.setParam('DisplayInterval', 1) # How often to log (e.g., 1 second) [2]

		# To write logs to a file as well
		#model.setParam('LogFile', 'gurobi.log') # Path to log file [2, 3]

		# optimize model
		model.optimize()
		
		tmp = 0
		if model.status == GRB.OPTIMAL:
			tmp = 1

		#xp_sol = [xp[i].X for i in range(N)]
		#xr_sol = [xr[i].X for i in range(N)]
		#sp_sol = [sp[i].X for i in range(N)]
		#sr_sol = [sr[i].X for i in range(N)]
		#yp_sol = [yp[i].X for i in range(N)]
		#yr_sol = [yr[i].X for i in range(N)]
	
		objval = model.ObjVal
		objbound = model.ObjBound 
		mipgap = model.MIPGap
		runtime = model.Runtime
		nodecount = model.NodeCount 

	except gp.GurobiError as e:
		print('Error code ' + str(e.errno) + ': ' + str(e))

	return objval, objbound, mipgap, runtime, nodecount, tmp


In [ ]:
result_path = Path('result/')

if __name__ == "__main__":

    # listar todos os arquivos .txt na pasta de instâncias
    #instance_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    # Ordenar numericamente após o prefixo '52_'
    #instance_files.sort(key=lambda x: int(x.split('_')[1].split('.')[0]))
    #instance_files = ['52_1.txt']
    #print(instance_files)
    
    #for datafile in instance_files:
    for it in range(1,2):
            datafile = f"52_{it}.txt"
            file_path = os.path.join(folder_path, datafile)
            if os.path.isfile(file_path): # garantir que é um arquivo
                N, H, P, F, D = readdata(datafile)
                objval, objbound, mipgap, runtime, nodecount, tmp = uls_std_mip(N, P, F, H, D)
                arquivo = open(os.path.join(result_path,'uls_std_mip.txt'),'a')
                arquivo.write(datafile+';'+str(round(objval,2))+';'+str(round(objbound,2))+';'+str(round(mipgap,2))+';'+str(round(runtime,2))+'\n')
                arquivo.close()

                start_time = time.time()
                sol_result = main_dynamic(N, H, P, F, D)
                end_time = time.time()
                run_time = end_time - start_time
                arquivo = open(os.path.join(result_path,'uls_dynamicp.txt'),'a')
                arquivo.write(datafile+';'+str(round(sol_result,2))+';'+str(round(run_time,2))+'\n')
                arquivo.close()

                # Verifica se o resultado é válido (não é float('inf'))
                #if file_result[-1] != float('inf'):
                #    results.append(file_result)
                #    total_time += file_result[-1]
                #else:
                #    print(f"Instância {datafile} falhou ou atingiu o limite de tempo interno.")
        #else:
        #    print(f"\nLimite de tempo ({time_limit}s) atingido. Interrompendo processamento.")
        #    break

    # criar um DataFrame com os resultados
    # Atualiza os nomes das colunas para corresponder aos 8 valores retornados por main_dynamic
    #results_df = pd.DataFrame(results, columns=['Instância', 'N', 'F', 'P', 'H', 'Demanda Total', 'Custo PD', 'Tempo PD (s)'])

Set parameter TimeLimit to value 360
Set parameter MIPGap to value 0.0001
Set parameter Threads to value 1
